# MODÈLE — `<source>` : `<table>` → landing (API, dlthub sur Databricks)

**Comment l'utiliser** (manuel 6, recette A) :
1. Importer ce notebook dans `ingestion/<source>/process/02_landing/01_loaders/` puis le renommer `<source>_<table>_l_loader`.
2. Chercher **`À ADAPTER`** (Ctrl + F) : 4 zones seulement (constantes, appels, extraction des lignes, contrôle d'erreur).
3. Les zones **`OPTIONNEL`** se gardent ou se suppriment (contrôle de quota API-Football, incrémental « déjà en landing »).
4. Lancer **Run all**, vérifier les sections 8 et 9, puis Commit & Push.

**Destination** : `/Volumes/<CATALOG>/landing/files/<source>/<table>/<aaaa>/<mm>/<jj>/<load_id>.<file_id>.parquet` (date du **dépôt**).

## 1. Paramètres (widgets)

`CATALOG` et `SECRET_SCOPE` sont toujours présents. Les autres widgets décrivent **ce qu'on demande à l'API** : À ADAPTER.

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")
dbutils.widgets.text("SECRET_SCOPE", "scope-data-dev", "Secret scope")
# À ADAPTER : widget métier (Open-Meteo géocodage : villes à géocoder)
dbutils.widgets.text("VILLES", "", "Villes (vide = toutes celles des fixtures)")   # test : Décines-Charpieu


## 2. Installation de dlthub (version fixée)

In [0]:
%pip install "dlt[parquet]==1.31.0"

In [0]:
%restart_python

## 3. Import de dlthub + contrôle « fail fast »

In [0]:
import dlt

if not hasattr(dlt, "__version__") or dlt.__file__.startswith("/databricks/"):
    raise ImportError("Conflit : le module 'dlt' chargé n'est pas dlthub (voir manuel 6, pièges)")
print("✅ dlthub", dlt.__version__)

## 4. Configuration

Toutes les valeurs propres à la source sont regroupées ici. **Aucune clé dans le code** : elle vient du secret scope.

In [0]:
import os, time
os.environ["RUNTIME__DLTHUB_TELEMETRY"] = "false"   # pas d'envoi de statistiques à dlthub
from dlt.sources.helpers import requests            # requests avec relances automatiques

def liste_entiers(texte: str) -> list[int]:
    """'61, 39,78' -> [61, 39, 78]"""
    return [int(x) for x in texte.split(",") if x.strip()]

CATALOG      = dbutils.widgets.get("CATALOG").strip()
SECRET_SCOPE = dbutils.widgets.get("SECRET_SCOPE").strip()

# ---------- À ADAPTER : identité de la source et de la table ----------
SOURCE   = "open_meteo"                         # nom du dossier ingestion/<source> (minuscules, _)
TABLE    = "geocodage"                             # nom de la table (minuscules, _)
BASE_URL = "https://geocoding-api.open-meteo.com"    # adresse de l'API
ENDPOINT = "/v1/search"                            # chemin appelé
SECRET_KEY_NAME = None              # nom de la clé dans le scope ; None si l'API est sans clé
HEADER_NAME     = None               # en-tête qui porte la clé (voir la doc de l'API)
PAUSE_SECONDES  = 1                               # pause entre deux appels (limite par minute de l'API)
TYPES_IMPOSES = {}
# ----------------------------------------------------------------------

HEADERS = {}
if SECRET_KEY_NAME:
    HEADERS[HEADER_NAME] = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_NAME)

LANDING = f"/Volumes/{CATALOG}/landing/files"
print(f"Catalog={CATALOG} | Source={SOURCE} | Table={TABLE} | Landing={LANDING}/{SOURCE}/{TABLE}/")

## 5. Liste des appels à faire

Un **appel** = un dictionnaire de paramètres envoyé à l'API. À ADAPTER selon la source.

on appelle ici toutes les villes qui sont présente dans api_football_fixture

In [0]:
VILLES_SAISIES = [v.strip() for v in dbutils.widgets.get("VILLES").split(",") if v.strip()]
if VILLES_SAISIES:                                   # mode test : seulement les villes saisies
    villes = VILLES_SAISIES
else:                                                # mode normal : toutes les villes des stades (bronze fixtures)
    villes = [r["ville"] for r in spark.sql(f"""
        SELECT DISTINCT fixture__venue__city AS ville
        FROM {CATALOG}.bronze.api_football_fixtures
        WHERE fixture__venue__city IS NOT NULL
    """).collect()]
APPELS = [{"name": v, "count": 1} for v in sorted(villes)]

## 5 bis. OPTIONNEL — Incrémental : ne pas rappeler ce qui est déjà complet en landing

Exemple API-Football : un couple championnat × saison est complet si tous ses matchs ont un statut final. À adapter (colonnes, règle) ou à **supprimer**.

In [0]:
# OPTIONNEL adapté à open_meteo géocodage
# Règle : une ville déjà présente en landing n'est PAS rappelée (ses coordonnées ne changent pas).
# On ESSAIE de lire plutôt que tester os.path.isdir : le cache du montage /Volumes peut mentir
# (dossier supprimé ailleurs mais encore vu comme présent) -> Spark répond PATH_NOT_FOUND.

CHEMIN_TABLE = f"{LANDING}/{SOURCE}/{TABLE}"                         # landing open_meteo/geocodage

def absent(erreur: Exception) -> bool:
    """True si l'erreur Spark signifie seulement « aucun fichier à ce chemin »."""
    return "PATH_NOT_FOUND" in str(erreur)

# Villes déjà présentes en landing (colonne _param_name = ville exacte envoyée à l'API)
deja_charges = set()
try:
    deja = (
        spark.read.parquet(f"{CHEMIN_TABLE}/*/*/*/*.parquet")
        .select("_param_name").distinct()
        .collect()
    )
    deja_charges = {r["_param_name"] for r in deja}
except Exception as e:
    if not absent(e):
        raise                                                        # autre erreur : on s'arrête
    print(f"ℹ️ Aucun fichier en landing pour {TABLE} (1er lancement ou dossier vidé)")

APPELS = [a for a in APPELS if a["name"] not in deja_charges]
print(f"Déjà en landing : {len(deja_charges)} | Appels restants : {len(APPELS)}")
if not APPELS:
    dbutils.notebook.exit("Rien à charger : toutes les villes demandées sont déjà en landing.")

## 7. La ressource dlthub

Une fonction qui **produit** des lignes (`yield`). Chaque ligne reçoit aussi les paramètres de son appel (colonnes `_param_<nom>`), pour savoir d'où elle vient.

In [0]:
def extraire_lignes(reponse_json):
    # Open-Meteo géocodage : résultats dans "results" ; clé ABSENTE si la ville n'est pas trouvée -> 0 ligne
    lignes = reponse_json.get("results", [])
    for ligne in lignes:
        if "postcodes" in ligne:                               # liste -> texte : évite une table enfant
            ligne["postcodes"] = ", ".join(ligne["postcodes"])
    return lignes

def verifier_erreurs(reponse_json, params):
    # Open-Meteo signale une erreur par "error": true + "reason"
    if reponse_json.get("error"):
        raise RuntimeError(f"Erreur API ({params}) : {reponse_json.get('reason')}")

@dlt.resource(name=TABLE, write_disposition="append", columns=TYPES_IMPOSES)   # types imposés (vide pour l'instant)
def ressource(appels: list[dict]):
    for i, params in enumerate(appels):
        if i > 0:
            time.sleep(PAUSE_SECONDES)                     # pause entre deux appels
        reponse_json = requests.get(f"{BASE_URL}{ENDPOINT}", headers=HEADERS, params=params).json()
        verifier_erreurs(reponse_json, params)
        lignes = extraire_lignes(reponse_json)
        print(f"📥 {params} : {len(lignes)} ligne(s)")
        yield [{**ligne, **{f"_param_{k}": v for k, v in params.items()}} for ligne in lignes]

## 8. Pipeline et chargement

`layout` = convention figée `<table>/<aaaa>/<mm>/<jj>/` (dlthub impose `{table_name}` avant la date).

In [0]:
pipeline = dlt.pipeline(
    pipeline_name=f"{SOURCE}_{TABLE}",
    destination=dlt.destinations.filesystem(
        bucket_url=f"file://{LANDING}",
        layout="{table_name}/{YYYY}/{MM}/{DD}/{load_id}.{file_id}.{ext}",
    ),
    dataset_name=SOURCE,
)
info = pipeline.run(ressource(APPELS), loader_file_format="parquet")
print(info)
info.raise_on_failed_jobs()

## 9. Vérifications : fichiers du jour et contenu

In [0]:
from datetime import datetime, timezone
dossier_jour = f"{LANDING}/{SOURCE}/{TABLE}/{datetime.now(timezone.utc):%Y/%m/%d}"
for f in sorted(os.listdir(dossier_jour)):
    print("📄", f)

df = spark.read.parquet(dossier_jour)
print("Colonnes :", len(df.columns), "| Lignes (dépôts du jour) :", df.count())
display(df.limit(20))

In [0]:
# TEMPORAIRE : villes des fixtures que l'API n'a pas trouvées (aucune ligne en landing)
trouvees = {r["_param_name"] for r in spark.read.parquet(f"{CHEMIN_TABLE}/*/*/*/*.parquet").select("_param_name").distinct().collect()}
non_trouvees = sorted(set(villes) - trouvees)        # villes = liste complète construite en cellule 5
print(len(non_trouvees), "ville(s) non trouvée(s) :")
for v in non_trouvees:
    print("  -", v)